# Entraînement des modèles

On va entraîner 3 modèles de régression pour prédire la puissance en watts.

Les modèles :
1. **Régression linéaire** — modèle de base simple
2. **Random Forest** — modèle ensembliste plus puissant
3. **XGBoost** — gradient boosting, généralement le meilleur sur ce type de données

On sépare les données en train/test en gardant les **20 dernières sorties** pour le test.
C'est plus réaliste que de mélanger aléatoirement : en pratique, on entraîne sur le passé
et on prédit le futur.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

print('Imports OK')

Imports OK


In [2]:
# Chargement du dataset préparé
df = pd.read_csv('../data/processed_dataset.csv')
print(f'Dataset chargé : {df.shape}')
df.head()

/var/folders/c3/_j7tnt4n6ms0s3bgh8r9xvr00000gn/T/ipykernel_71693/3286220339.py:2: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/processed_dataset.csv')


Dataset chargé : (3240910, 15)


,secs,km,power,hr,cad,alt,ride_id,vitesse_kmh,pente_pct,acceleration,delta_hr,vitesse_moy_5s,hr_moy_5s,cad_moy_5s,pente_moy_5s
0,0,0.00000,0,90,0,11.2,1,0.000,0.0,0.000,0.0,0.0000,90.0,0.0,0.0
1,1,0.00348,0,91,0,11.2,1,12.528,0.0,10.000,1.0,6.2640,90.5,0.0,0.0
2,2,0.00675,0,92,0,11.2,1,11.772,0.0,-0.756,1.0,8.1000,91.0,0.0,0.0
3,3,0.00978,0,93,0,11.2,1,10.908,0.0,-0.864,1.0,8.8020,91.5,0.0,0.0
4,4,0.01312,0,95,0,11.2,1,12.024,0.0,1.116,2.0,9.4464,92.2,0.0,0.0


In [3]:
# Définition des features et de la cible
# On n'utilise pas 'secs', 'km' (pas informatifs pour la puissance)
# ni 'ride_id' (c'est juste un identifiant)
features = ['hr', 'cad', 'alt', 'vitesse_kmh', 'pente_pct',
            'acceleration', 'delta_hr', 'vitesse_moy_5s',
            'hr_moy_5s', 'cad_moy_5s', 'pente_moy_5s']

cible = 'power'

print('Features utilisées :', features)
print('Variable cible :', cible)

Features utilisées : ['hr', 'cad', 'alt', 'vitesse_kmh', 'pente_pct', 'acceleration', 'delta_hr', 'vitesse_moy_5s', 'hr_moy_5s', 'cad_moy_5s', 'pente_moy_5s']
Variable cible : power


In [ ]:
# Séparation train / test
# On garde les 20 dernières sorties pour le test (sorties 144 à 163)
# et on entraîne sur les 143 premières

# On filtre les ride_id non-numériques (au cas où processed_dataset.csv
# se serait glissé dans les données lors d'une re-exécution)
tous_les_rides = sorted(
    [r for r in df['ride_id'].unique() if str(r).isdigit()],
    key=lambda x: int(x)
)
df = df[df['ride_id'].isin(tous_les_rides)]

rides_test = tous_les_rides[-20:]   # les 20 dernières sorties
rides_train = tous_les_rides[:-20]  # toutes les autres

print(f'Sorties en entraînement : {len(rides_train)}')
print(f'Sorties en test : {len(rides_test)}')

df_train = df[df['ride_id'].isin(rides_train)]
df_test  = df[df['ride_id'].isin(rides_test)]

X_train = df_train[features]
y_train = df_train[cible]
X_test  = df_test[features]
y_test  = df_test[cible]

print(f'\nTrain : {X_train.shape[0]} lignes')
print(f'Test  : {X_test.shape[0]} lignes')

In [ ]:
# --- Modèle 1 : Régression linéaire ---
print('Entraînement de la régression linéaire...')
modele_lineaire = LinearRegression()
modele_lineaire.fit(X_train, y_train)

pred_lineaire = modele_lineaire.predict(X_test)
r2_lin = r2_score(y_test, pred_lineaire)
mae_lin = mean_absolute_error(y_test, pred_lineaire)
rmse_lin = np.sqrt(mean_squared_error(y_test, pred_lineaire))

print(f'  R²   : {r2_lin:.4f}')
print(f'  MAE  : {mae_lin:.2f} W')
print(f'  RMSE : {rmse_lin:.2f} W')

In [ ]:
# --- Modèle 2 : Random Forest ---
# On utilise seulement 50 arbres pour que l'entraînement reste rapide
print('Entraînement du Random Forest...')
modele_rf = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
modele_rf.fit(X_train, y_train)

pred_rf = modele_rf.predict(X_test)
r2_rf = r2_score(y_test, pred_rf)
mae_rf = mean_absolute_error(y_test, pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, pred_rf))

print(f'  R²   : {r2_rf:.4f}')
print(f'  MAE  : {mae_rf:.2f} W')
print(f'  RMSE : {rmse_rf:.2f} W')

In [ ]:
# --- Modèle 3 : XGBoost ---
print('Entraînement de XGBoost...')
modele_xgb = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1, verbosity=0)
modele_xgb.fit(X_train, y_train)

pred_xgb = modele_xgb.predict(X_test)
r2_xgb = r2_score(y_test, pred_xgb)
mae_xgb = mean_absolute_error(y_test, pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, pred_xgb))

print(f'  R²   : {r2_xgb:.4f}')
print(f'  MAE  : {mae_xgb:.2f} W')
print(f'  RMSE : {rmse_xgb:.2f} W')

In [ ]:
# Tableau comparatif des 3 modèles
resultats = pd.DataFrame({
    'Modèle': ['Régression Linéaire', 'Random Forest', 'XGBoost'],
    'R²': [r2_lin, r2_rf, r2_xgb],
    'MAE (W)': [mae_lin, mae_rf, mae_xgb],
    'RMSE (W)': [rmse_lin, rmse_rf, rmse_xgb]
})
resultats = resultats.round(4)
print(resultats.to_string(index=False))

In [ ]:
# Graphique : valeurs réelles vs prédites pour XGBoost
echantillon = np.random.choice(len(y_test), size=3000, replace=False)
y_reel = np.array(y_test)[echantillon]
y_predit = pred_xgb[echantillon]

plt.figure(figsize=(8, 8))
plt.scatter(y_reel, y_predit, alpha=0.2, s=5, color='steelblue')
plt.plot([0, 1000], [0, 1000], 'r--', linewidth=1, label='Prédiction parfaite')
plt.title(f'XGBoost : Réel vs Prédit (R²={r2_xgb:.3f})')
plt.xlabel('Puissance réelle (W)')
plt.ylabel('Puissance prédite (W)')
plt.legend()
plt.tight_layout()
plt.savefig('../plots/xgboost_reel_vs_predit.png')
plt.show()

In [ ]:
# Importance des features pour XGBoost
importances = pd.Series(modele_xgb.feature_importances_, index=features).sort_values(ascending=True)

plt.figure(figsize=(8, 6))
importances.plot(kind='barh', color='steelblue')
plt.title('Importance des features - XGBoost')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig('../plots/feature_importance.png')
plt.show()

In [ ]:
# Sauvegarde des modèles
joblib.dump(modele_lineaire, '../models/linear_regression.joblib')
joblib.dump(modele_rf,       '../models/random_forest.joblib')
joblib.dump(modele_xgb,      '../models/xgboost.joblib')

print('Modèles sauvegardés dans models/')
print('  - linear_regression.joblib')
print('  - random_forest.joblib')
print('  - xgboost.joblib')

## Conclusions

- La **régression linéaire** donne un R² de base — bon point de comparaison
- Le **Random Forest** améliore nettement les résultats
- **XGBoost** est le meilleur des trois modèles

Les features les plus importantes sont la vitesse, la cadence et la fréquence cardiaque.

**Prochaine étape** : brancher ces modèles dans le template (`src/config.py`, `src/data.py`, `src/metrics.py`).